# Exercise 4 - YSocial Sandbox

Pick a focal agent and inspect their content and exposure with explicit `ysights` calls.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / ".cache"
PLOTS_DIR = CACHE_DIR / "plots"
(CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import sqlite3

import pandas as pd
from ysights import YDataHandler


In [ ]:
def ensure_ysocial_demo_db(db_path=DATA_RAW / "ysocial_demo.sqlite", seed=42):
    if Path(db_path).exists():
        return Path(db_path)

    rng = np.random.default_rng(seed)
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()

    cursor.executescript(
        '''
        CREATE TABLE rounds (id INTEGER PRIMARY KEY, day INTEGER, hour INTEGER);
        CREATE TABLE user_mgmt (
            id INTEGER PRIMARY KEY,
            username TEXT,
            avatar TEXT,
            bio TEXT,
            role TEXT,
            leaning TEXT,
            age INTEGER,
            oe REAL,
            co REAL,
            ex REAL,
            ag REAL,
            ne REAL,
            content_recsys REAL,
            language TEXT,
            region TEXT,
            education TEXT,
            joined INTEGER,
            social_recsys REAL,
            gender TEXT,
            nationality TEXT,
            toxicity REAL,
            is_page INTEGER,
            left_on INTEGER,
            daily_activity_level REAL,
            profession TEXT
        );
        CREATE TABLE follow (user_id INTEGER, follower_id INTEGER, action TEXT, round INTEGER);
        CREATE TABLE post (
            id INTEGER PRIMARY KEY,
            text TEXT,
            media TEXT,
            user_id INTEGER,
            comment_to INTEGER,
            thread_id INTEGER,
            round INTEGER,
            news_id INTEGER,
            shared_from INTEGER,
            image_id INTEGER
        );
        CREATE TABLE mentions (post_id INTEGER, user_id INTEGER);
        CREATE TABLE hashtags (id INTEGER PRIMARY KEY, hashtag TEXT);
        CREATE TABLE post_hashtags (post_id INTEGER, hashtag_id INTEGER);
        CREATE TABLE interests (iid INTEGER PRIMARY KEY, interest TEXT);
        CREATE TABLE user_interest (user_id INTEGER, interest_id INTEGER, round INTEGER);
        CREATE TABLE post_topics (post_id INTEGER, topic_id INTEGER);
        CREATE TABLE emotions (id INTEGER PRIMARY KEY, emotion TEXT);
        CREATE TABLE post_emotions (post_id INTEGER, emotion_id INTEGER);
        CREATE TABLE post_toxicity (
            post_id INTEGER PRIMARY KEY,
            model TEXT,
            toxicity REAL,
            severe_toxicity REAL,
            identity_attack REAL,
            insult REAL,
            profanity REAL,
            threat REAL,
            sexual_explicit REAL,
            flirtation REAL
        );
        '''
    )

    for round_id in range(24):
        cursor.execute("INSERT INTO rounds (id, day, hour) VALUES (?, ?, ?)", (round_id, 1 + round_id // 12, round_id % 12))

    for hashtag_id, hashtag in {1: "#civicdata", 2: "#community", 3: "#publichealth", 4: "#marketfreedom", 5: "#innovation", 6: "#trustednews"}.items():
        cursor.execute("INSERT INTO hashtags (id, hashtag) VALUES (?, ?)", (hashtag_id, hashtag))
    for interest_id, interest in {1: "health", 2: "governance", 3: "innovation", 4: "markets", 5: "media"}.items():
        cursor.execute("INSERT INTO interests (iid, interest) VALUES (?, ?)", (interest_id, interest))
    for emotion_id, emotion in {1: "joy", 2: "anger", 3: "fear", 4: "hope"}.items():
        cursor.execute("INSERT INTO emotions (id, emotion) VALUES (?, ?)", (emotion_id, emotion))

    agents = []
    for agent_id in range(1, 21):
        leaning = "left" if agent_id <= 10 else "right"
        enclave = agent_id in {3, 4, 5, 13, 14, 15}
        profession = "journalist" if agent_id % 5 == 0 else "researcher"
        agents.append(
            (
                agent_id, f"agent_{agent_id:02d}", None, None, "user", leaning, int(rng.integers(22, 60)),
                round(float(rng.uniform(0.3, 0.9)), 3), round(float(rng.uniform(0.3, 0.9)), 3),
                round(float(rng.uniform(0.3, 0.9)), 3), round(float(rng.uniform(0.3, 0.9)), 3),
                round(float(rng.uniform(0.3, 0.9)), 3), 0.85 if enclave else 0.35, "en", "EU", "graduate",
                0, 0.8 if enclave else 0.4, "female" if agent_id % 2 == 0 else "male",
                "Kenya" if agent_id % 3 == 0 else "Italy", 0.45 if enclave else 0.18, 0, None,
                2.5 if enclave else 1.7, profession,
            )
        )

    cursor.executemany(
        '''
        INSERT INTO user_mgmt (
            id, username, avatar, bio, role, leaning, age, oe, co, ex, ag, ne,
            content_recsys, language, region, education, joined, social_recsys,
            gender, nationality, toxicity, is_page, left_on, daily_activity_level, profession
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''',
        agents,
    )

    follow_edges = [
        (1, 2), (1, 3), (1, 6), (2, 3), (2, 4), (3, 4), (3, 5), (4, 5), (4, 7), (5, 8),
        (6, 1), (6, 2), (7, 1), (7, 3), (8, 4), (8, 5), (9, 2), (9, 10), (10, 1), (10, 6),
        (11, 12), (11, 13), (11, 16), (12, 13), (12, 14), (13, 14), (13, 15), (14, 15), (14, 17), (15, 18),
        (16, 11), (16, 12), (17, 11), (17, 13), (18, 14), (18, 15), (19, 12), (19, 20), (20, 11), (20, 16),
        (6, 11), (10, 12), (16, 1), (17, 2),
    ]
    for round_id, (follower_id, user_id) in enumerate(follow_edges, start=1):
        cursor.execute(
            "INSERT INTO follow (user_id, follower_id, action, round) VALUES (?, ?, ?, ?)",
            (user_id, follower_id, "follow", round_id % 24),
        )

    post_id = 1
    for round_id in range(24):
        for user_id in rng.choice(np.arange(1, 21), size=4, replace=False):
            leaning = "left" if user_id <= 10 else "right"
            enclave = user_id in {3, 4, 5, 13, 14, 15}
            hashtags = [1, 6] if leaning == "left" else [4, 5]
            hashtags.append(2 if not enclave else 3)
            topic_id = 2 if leaning == "left" else 4
            if enclave:
                topic_id = 5
            emotion_id = 2 if enclave else 4
            mention_target = int(rng.choice(np.arange(1, 11) if leaning == "left" else np.arange(11, 21)))
            if round_id % 7 == 0:
                mention_target = 12 if leaning == "left" else 2

            cursor.execute(
                '''
                INSERT INTO post (id, text, media, user_id, comment_to, thread_id, round, news_id, shared_from, image_id)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''',
                (
                    post_id,
                    f"Round {round_id}: {'solidarity' if leaning == 'left' else 'competition'} and platform curation shape what we see.",
                    None,
                    int(user_id),
                    None,
                    round_id,
                    round_id,
                    None,
                    None,
                    None,
                ),
            )
            cursor.execute("INSERT INTO mentions (post_id, user_id) VALUES (?, ?)", (post_id, mention_target))
            for hashtag_id in hashtags:
                cursor.execute("INSERT INTO post_hashtags (post_id, hashtag_id) VALUES (?, ?)", (post_id, hashtag_id))
            cursor.execute("INSERT INTO post_topics (post_id, topic_id) VALUES (?, ?)", (post_id, topic_id))
            cursor.execute("INSERT INTO post_emotions (post_id, emotion_id) VALUES (?, ?)", (post_id, emotion_id))
            cursor.execute(
                '''
                INSERT INTO post_toxicity (
                    post_id, model, toxicity, severe_toxicity, identity_attack,
                    insult, profanity, threat, sexual_explicit, flirtation
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''',
                (post_id, "demo", 0.42 if enclave else 0.17, 0.1 if enclave else 0.03, 0.05 if enclave else 0.02, 0.18 if enclave else 0.05, 0.12 if enclave else 0.04, 0.02, 0.0, 0.0),
            )
            post_id += 1

    for agent_id in range(1, 21):
        leaning = "left" if agent_id <= 10 else "right"
        interest_ids = [1, 2, 5] if leaning == "left" else [3, 4, 5]
        for round_id, interest_id in enumerate(interest_ids, start=1):
            cursor.execute("INSERT INTO user_interest (user_id, interest_id, round) VALUES (?, ?, ?)", (agent_id, interest_id, round_id + agent_id))

    connection.commit()
    connection.close()
    return Path(db_path)


In [ ]:
DB_PATH = ensure_ysocial_demo_db()
handler = YDataHandler(str(DB_PATH))

focal_agent = 3
pd.Series(
    {
        "hashtags": handler.agent_hashtags(focal_agent),
        "interests": handler.agent_interests(focal_agent),
        "posts": len(handler.posts_by_agent(focal_agent, enrich_dimensions=[]).get_posts()),
    }
)
